<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L28_RAG_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L28: RAG Systems - Retrieval-Augmented Generation

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 28 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Build a minimal RAG pipeline: retriever + generator
- Embed documents and queries; retrieve by similarity
- Pass retrieved context to the LM and generate an answer

---

## Concept: RAG

**RAG** = retrieve relevant documents (or chunks) for a query, then condition the LM on that context to generate an answer. Reduces hallucination and keeps knowledge up-to-date. Steps: (1) Index documents (embed and store); (2) Embed query; (3) Retrieve top-k chunks; (4) Concatenate context + query; (5) Generate.

---

In [ ]:
!pip install transformers torch sentence-transformers -q
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Document Store and Embeddings

---

In [ ]:
documents = [
    "Paris is the capital of France. It is known for the Eiffel Tower.",
    "Machine learning is a subset of AI that learns from data.",
    "The Eiffel Tower was built in 1889 and is in Paris.",
    "Python is a programming language used for ML and data science.",
]

encoder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = encoder.encode(documents)
print(f"Embeddings shape: {doc_embeddings.shape}")

## Part 2: Retrieve by Similarity

---

In [ ]:
import numpy as np

def retrieve(query, doc_embeddings, documents, top_k=2):
    q_emb = encoder.encode([query])
    scores = np.dot(doc_embeddings, q_emb.T).squeeze()
    top_idx = np.argsort(scores)[-top_k:][::-1]
    return [documents[i] for i in top_idx]

context = retrieve("Where is the Eiffel Tower?", doc_embeddings, documents)
print("Retrieved:", context)

## Part 3: Generate Answer with Context

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

query = "Where is the Eiffel Tower?"
context_str = " ".join(retrieve(query, doc_embeddings, documents))
prompt = f"Context: {context_str}\n\nQuestion: {query}\nAnswer:"

out = generator(prompt, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id)
answer = out[0]["generated_text"][len(prompt):].strip()
print("Answer:", answer)

## Exercises

1. Add more documents and try different queries; tune top_k.
2. Chunk long documents (e.g. by sentence or 256 tokens) and index chunks.
3. Use a vector DB (e.g. FAISS, L29) for scalable retrieval.

---

## Key Takeaways

1. RAG = retrieve (embed + similarity) + generate (context + query → LM).
2. Good retrieval (embedding model, chunking) is as important as the generator.
3. RAG reduces hallucination by grounding in retrieved text.

---

## Next Lesson

**L29: Vector Databases** – FAISS, embeddings, and semantic search at scale.

---